In [25]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [26]:
df = pd.read_csv("Student_Data.csv")

print(df.shape)
df.head()

(2000, 18)


,id,sgpa1,sgpa2,sgpa3,sgpa4,cgpa,active_backlogs,total_backlogs,10th_mark,12th_mark,studytime,part_timejob,math,dsa,networks,os,attendance,career_preference
0,1,6.79,7.94,7.12,6.22,7.02,0,0,81.91,84.23,13,no,52,88,90,85,85.93,higher study
1,2,9.82,5.18,7.00,6.27,7.07,0,0,45.37,33.45,10,yes,37,34,81,81,71.06,job
2,3,6.74,6.52,6.53,7.96,6.94,0,0,56.23,64.97,0,yes,50,48,50,86,70.80,higher study
3,4,8.72,8.62,4.71,7.46,7.38,0,0,77.44,86.19,2,yes,97,58,77,88,97.16,job
4,5,6.75,7.84,7.82,8.37,7.70,4,5,65.31,75.22,2,yes,55,53,37,71,72.13,none


In [27]:
# Drop id if present
if "id" in df.columns:
    df = df.drop(columns=["id"])

# Encode part_timejob
if df["part_timejob"].dtype == "object":
    df["part_timejob"] = df["part_timejob"].map({"yes":1,"no":0})

# Encode career_preference
le = LabelEncoder()
df["career_preference"] = le.fit_transform(df["career_preference"])

In [28]:
df["avg_sgpa_4"] = (df["sgpa1"] + df["sgpa2"] + df["sgpa3"] + df["sgpa4"]) / 4

df["sgpa_trend"] = df["sgpa4"] - df["sgpa1"]

df["recent_trend"] = df["sgpa4"] - df["sgpa3"]

df["academic_score"] = (
    df["math"] + df["dsa"] + df["networks"] + df["os"]
) / 4

In [29]:
df["sgpa5"] = (
    df["avg_sgpa_4"]
    + 0.4 * df["recent_trend"]
    + 0.2 * df["sgpa_trend"]
    + 0.01 * (df["attendance"] - 75)
    + 0.005 * (df["academic_score"] - 70)
)

df["sgpa5"] = df["sgpa5"].clip(0,10)

In [30]:
X = df.drop(columns=["sgpa5"])
y = df["sgpa5"]

print("Features:", X.shape)

Features: (2000, 21)


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, "scaler.pkl")

print("Scaler saved")

Scaler saved


In [33]:
model = Sequential()

model.add(Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation="relu"))
model.add(Dense(16, activation="relu"))
model.add(Dense(1))

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

c:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,033 (15.75 KB)

 Trainable params: 4,033 (15.75 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 24.3194 - mae: 4.2374 - val_loss: 2.8485 - val_mae: 1.2658
Epoch 2/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.6634 - mae: 1.0196 - val_loss: 1.2702 - val_mae: 0.8875
Epoch 3/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1286 - mae: 0.8553 - val_loss: 1.0150 - val_mae: 0.8046
Epoch 4/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8878 - mae: 0.7549 - val_loss: 0.8388 - val_mae: 0.7322
Epoch 5/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7157 - mae: 0.6774 - val_loss: 0.7257 - val_mae: 0.6843
Epoch 6/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6002 - mae: 0.6201 - val_loss: 0.6208 - val_mae: 0.6325
Epoch 7/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5013 - mae: 0.5619 - val_loss: 0.5552 - val_mae: 0.5896
Epoch 8/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4262 - mae: 0.5163 - val_loss: 0.4880 - val_mae: 0.5571
Epoch 9/150
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.37

In [35]:
loss, mae = model.evaluate(X_test, y_test)

print("Test MAE:", mae)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0119 - mae: 0.0817 
Test MAE: 0.08166773617267609


In [36]:
model.save("sgpa5_prediction_model.h5")

print("Model saved as sgpa5_prediction_model.h5")

Model saved as sgpa5_prediction_model.h5


In [37]:
loaded_model = load_model("sgpa5_prediction_model.h5", compile=False)

print("Model loaded successfully")

Model loaded successfully


In [39]:
predictions = loaded_model.predict(X_test)
print(predictions[:5])

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
[[7.301551 ]
 [6.838526 ]
 [6.076255 ]
 [6.4996977]
 [8.643383 ]]


In [40]:
predictions = loaded_model.predict(X_test)
predictions = predictions.flatten()

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [41]:
X_test_df = pd.DataFrame(X_test, columns=X.columns)

In [42]:
X_test_df["predicted_sgpa5"] = predictions

X_test_df.head(20)

,sgpa1,sgpa2,sgpa3,sgpa4,cgpa,active_backlogs,total_backlogs,10th_mark,12th_mark,studytime,...,dsa,networks,os,attendance,career_preference,avg_sgpa_4,sgpa_trend,recent_trend,academic_score,predicted_sgpa5
0,0.014038,0.387779,-1.460095,-0.000690,-0.532389,2.372297,1.386128,0.506361,-1.738644,-0.497263,...,0.407236,0.843463,-1.104959,-1.596804,0.011481,-0.528720,-0.010364,1.039696,0.238661,7.301551
1,-0.552849,1.890242,-0.654508,-0.724813,-0.023274,2.372297,2.683848,0.335292,1.383465,-0.267182,...,0.562706,-0.741525,-0.183627,-0.980082,0.011481,-0.019570,-0.121454,-0.056425,-0.421333,6.838526
2,1.707523,0.394967,-0.015844,-1.025934,0.529480,1.445845,2.034988,0.543349,-0.011687,0.883224,...,1.236412,-0.230239,-1.053774,0.546042,0.011481,0.536858,-1.924139,-0.728575,-0.522871,6.076255
3,-1.464174,-0.359859,-0.444039,-0.739152,-1.521528,3.298750,2.034988,1.483460,1.034938,-0.267182,...,-1.510235,0.587820,1.351928,1.431054,0.011481,-1.521563,0.509738,-0.216707,-0.142105,6.499698
4,-0.732244,0.589066,-0.581932,0.859654,0.078549,-0.407060,-0.560453,-0.851406,-0.332519,-0.727344,...,-1.147470,-1.712970,1.710224,1.209801,0.011481,0.071349,1.120733,1.034526,0.162508,8.643383
5,-0.739420,1.279192,1.138105,-0.696135,0.485842,-0.407060,-0.560453,0.295735,0.994181,-1.417588,...,1.547353,0.843463,-1.156145,-0.832000,0.011481,0.489580,0.030032,-1.312828,0.543273,6.415446
6,-0.538498,0.962884,0.405093,0.350617,0.587665,0.519392,0.088407,-1.476606,-1.380712,-0.037101,...,0.251765,1.405878,-0.644293,-0.483569,1.236115,0.595046,0.625878,-0.035744,0.289430,7.956289
7,0.896659,-0.712111,-0.654508,0.902671,0.224011,-0.407060,-0.560453,1.668914,-0.662235,0.192981,...,-1.354764,-0.946040,0.737706,0.302141,0.011481,0.224094,0.004784,1.117252,-1.538246,8.132633
8,-0.495443,1.178549,0.891349,1.081910,1.344065,-0.407060,-0.560453,-0.267304,-0.835714,-0.497263,...,-0.940176,-0.281367,1.761409,0.372698,-1.213153,1.340587,1.110633,0.145220,1.076346,8.870778
9,-0.409334,1.796788,-0.277116,-0.660287,0.224011,-0.407060,-0.560453,1.585691,0.616393,0.883224,...,0.199942,0.894592,1.147187,0.444126,-1.213153,0.227731,-0.176999,-0.278751,0.898655,7.153145


In [ ]:
# X_test_df.to_csv("sgpa5_predicted_dataset.csv", index=False)

# print("Dataset saved")